In [23]:
# Load env variables and create client
import json
import ast
import re
import json
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [ ]:
#helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 500,
        "messages": messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [26]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks.

Generate an array of JSON objects. Each object must contain:
- "task": a description of the task
- "format": either "python", "json", or "regex"

Example output:
```json
[
  {
    "task": "Create a Python function to validate an AWS IAM username",
    "format": "python"
  }
]

Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex.

Focus on tasks that do not require writing much code.

Please generate 3 objects.
"""

    messages = []

    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")

    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [27]:
dataset = generate_dataset()
print(dataset)

[{'task': 'Create a regex pattern to validate an AWS S3 bucket name', 'format': 'regex'}, {'task': 'Generate a JSON policy document that allows read-only access to an S3 bucket', 'format': 'json'}, {'task': 'Create a Python function to parse an AWS ARN string and extract the service, account ID, and resource', 'format': 'python'}]


In [28]:
with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
# Passes a test case into Claude

def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [36]:
# Function to execute a single test case and grade the output
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    model_grade = grade_by_model(test_case, output)


    model_score = model_grade["score"]

    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "model_score": model_score,
        "syntax_score": syntax_score,
        "score": score,
        "reasoning": model_grade["reasoning"]
    }

In [38]:
from statistics import mean

def run_eval(dataset):
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean([result["score"] for result in results])

    print(f"Average score: {average_score}")

    return results

In [39]:
import json

with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [40]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Bucket ARN Region Extractor\n\nHere's a comprehensive solution:\n\n```python\ndef extract_region_from_s3_arn(arn):\n    \"\"\"\n    Extract the AWS region from an S3 bucket ARN.\n    \n    S3 bucket ARNs have the format:\n    - arn:aws:s3:::bucket-name (global, no region)\n    - arn:aws:s3:region:account-id:bucket/bucket-name (object ARN with region)\n    \n    Args:\n        arn (str): The S3 bucket or object ARN\n        \n    Returns:\n        str or None: The AWS region if found, None otherwise\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        return None\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Valid S3 ARN format: arn:aws:s3:[region]:[account-id]:bucket[/object-path]\n    # S3 bucket ARNs typically don't have regions, but object ARNs might\n    if len(parts) >= 6:\n        # For object ARNs: arn:aws:s3:region:account-id:bucket/key\n        region = parts[3]\n        if region:  # Non-empty region s

In [ ]:
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert code reviewer.

Task: {test_case["task"]}

Solution:
{output}

Return ONLY valid JSON:

{{
  "strengths": ["short"],
  "weaknesses": ["short"],
  "reasoning": "one short sentence",
  "score": 1
}}

Rules:
- score: number from 1 to 10
- maximum 2 strengths
- maximum 2 weaknesses
- reasoning: one short sentence
- no explanation outside JSON
"""

    messages = []
    add_user_message(messages, eval_prompt)

    eval_text = chat(messages)

    print("GRADER RESPONSE:")
    print(repr(eval_text))

    return json.loads(eval_text.strip())

In [24]:
#functions to validate output formats for code grader
def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


In [25]:
def grade_syntax(output, test_case):
    format_type = test_case["format"]

    if format_type == "json":
        return validate_json(output)

    elif format_type == "python":
        return validate_python(output)

    elif format_type == "regex":
        return validate_regex(output)

    return 0